# Avaliação da aorta em bancos CCTA externos

Esta análise reúne a avaliação visual e o Dice da aorta em resoluções mid e high no MM-WHS e no OrCaScore. A comparação dos alertas automáticos nos três datasets e em todos os subconjuntos fica em [Avaliação automática da qualidade da aorta](../aorta/aorta_automatic_quality_assessment.ipynb). Os alertas permanecem aqui somente como contexto nas tabelas por exame. Os percentuais incluem os exames “Não avaliável” ou “Sem feedback” no denominador. As classificações não são uma validação clínica.

O Dice do MM-WHS usa comparação direta nos 20 exames de treino e o avaliador oficial com label 820 nos exames de teste. Os protocolos são apresentados separadamente.

Os resultados quantitativos vêm dos runs de 06/10/2026: mid `2026-10-06_07-53-03` e high `2026-10-06_08-33-19`. As avaliações visuais foram copiadas, sem alteração, dos respectivos runs de 22/09/2026: mid `2026-09-22_18-51-28` e high `2026-09-22_17-47-45`. São avaliações históricas reutilizadas, não uma nova inspeção dos runs atuais.

In [ ]:
# ruff: noqa: E402
import json
from importlib import reload
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import display
from nibabel.loadsave import load as load_nifti

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.analysis.external_aorta_assessment import (
    attach_aorta_feedback,
)
from utils.project.results.external import (
    load_mmwhs_train_aorta_dice,
    load_mmwhs_test_aorta_dice,
)
from utils.project.analysis.external_visual_assessment import (
    AORTA_ARTERY_STATUS_ORDER,
    attach_assessment_subsets,
    load_assessment_subset_lookup,
    load_external_visual_assessments,
    summarize_visual_status,
)

from utils.project.datasets.ccta import (
    discover_ccta_dataset,
    load_ccta_aorta_ground_truth,
)
from utils.project.evaluation.mmwhs_official_aorta import (
    predict_aorta,
    restore_native_mask,
    validate_saved_prediction,
)
from utils.project.runtime.notebook_env import resolve_existing_path
from utils.utils.roi import mask_bounding_box_slices
import utils.visualization.images.volume as volume_visualization

# Atualiza a função mesmo quando o kernel já importou uma versão anterior.
volume_visualization = reload(volume_visualization)
from utils.visualization.images.volume import (
    visualize_binary_masks_comparison,
    save_k3d_plot_html,
)

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

## Configuração e carregamento

Defina `EXTERNAL_RESULTS_ROOT` com a pasta que contém `mmwhs/` e `orcascore/`. A tabela abaixo identifica arquivos ausentes. As seções sem fontes são ignoradas com aviso; o Dice pode ser consultado sem as planilhas visuais, desde que os CSVs mid/high estejam disponíveis. Não há substituição automática por runs históricos.


In [ ]:
# O 3D é opcional: pode recalcular a aorta de um único exame de treino.
SHOW_QUALITATIVE = True
QUALITATIVE_RESOLUTION = "mid"
QUALITATIVE_SUBSET = "train"
QUALITATIVE_EXAM_ID = "ct_train_1007"  # Dice mid: 0,9338.
QUALITATIVE_REFERENCE_PATH = (
    None  # Referência pública/legível, nunca o label criptografado.
)
MESH_SMOOTHING_ITERATIONS = 2
MMWHS_DATA_ROOT = (
    Path(os.environ["MMWHS_BASE_PATH"]) if os.environ.get("MMWHS_BASE_PATH") else None
)

EXTERNAL_RESULTS_ROOT = Path(
    os.environ.get("EXTERNAL_CCTA_RESULTS_ROOT")
    or os.environ.get("CCTA_RESULTS_ROOT")
    or REPO_ROOT / "output/segmentation/external_ccta"
)
RUN_DIRS = {
    "mid": {
        "MM-WHS": EXTERNAL_RESULTS_ROOT / "mmwhs/mid_res/2026-10-06_07-53-03",
        "OrCaScore": EXTERNAL_RESULTS_ROOT / "orcascore/mid_res/2026-09-19_10-23-50",
    },
    "high": {
        "MM-WHS": EXTERNAL_RESULTS_ROOT / "mmwhs/high_res/2026-10-06_08-33-19",
        "OrCaScore": EXTERNAL_RESULTS_ROOT / "orcascore/high_res/2026-09-22_19-12-41",
    },
}
DATASET_ORDER = ["MM-WHS", "OrCaScore"]
VISUAL_COLORS = {
    "Adequada": "#22C55E",
    "Parcial": "#EAB308",
    "Inadequada": "#EF4444",
    "Não detectada": "#F97316",
    "Não avaliável": "#9CA3AF",
}
RESOLUTION_COLORS = {"mid": "#4C78A8", "high": "#F28E2B"}

# Verifica as fontes sem trocar runs científicos por referências históricas.
source_rows = []
for resolution, run_dirs in RUN_DIRS.items():
    for dataset, run_dir in run_dirs.items():
        for kind, relative_path in (
            ("Resultados", "numeric/results_all.csv"),
            ("Avaliação visual", "visual/avaliacao_visual.xlsx"),
        ):
            path = run_dir / relative_path
            source_rows.append(
                {
                    "Resolução": resolution,
                    "Banco": dataset,
                    "Fonte": kind,
                    "Disponível": path.is_file(),
                    "Caminho": str(path),
                }
            )
source_status = pd.DataFrame(source_rows)
display(source_status)
VISUAL_AVAILABLE = bool(source_status["Disponível"].all())
MMWHS_NUMERIC_AVAILABLE = all(
    (RUN_DIRS[resolution]["MM-WHS"] / "numeric/results_all.csv").is_file()
    for resolution in ("mid", "high")
)
if not source_status["Disponível"].all():
    print(
        "Fontes ausentes: monte o disco externo ou ajuste EXTERNAL_RESULTS_ROOT "
        "(variável EXTERNAL_CCTA_RESULTS_ROOT ou CCTA_RESULTS_ROOT). "
        "Nenhum run antigo será usado como substituto."
    )

assessments = {}
if VISUAL_AVAILABLE:
    subset_lookup = load_assessment_subset_lookup(
        {
            dataset: [
                RUN_DIRS[resolution][dataset] / "numeric/results_all.csv"
                for resolution in RUN_DIRS
            ]
            for dataset in DATASET_ORDER
        }
    )
    assessments = {
        resolution: attach_assessment_subsets(
            load_external_visual_assessments(
                {
                    dataset: run_dir / "visual/avaliacao_visual.xlsx"
                    for dataset, run_dir in run_dirs.items()
                }
            ),
            subset_lookup,
        )
        for resolution, run_dirs in RUN_DIRS.items()
    }

In [ ]:
feedback_by_resolution = {}
if VISUAL_AVAILABLE:
    feedback_by_resolution = {
        resolution: attach_aorta_feedback(
            assessments[resolution].loc[
                :, ["dataset", "image_id", "subset", "aorta_result"]
            ],
            {
                dataset: run_dir / "numeric/results_all.csv"
                for dataset, run_dir in run_dirs.items()
            },
        )
        for resolution, run_dirs in RUN_DIRS.items()
    }

In [ ]:
def show_aorta_distribution(counts, title, colors, show_counts=False):
    counts = counts.reindex(index=DATASET_ORDER, fill_value=0)
    counts.loc["Geral"] = counts.sum(axis=0)
    totals = counts.sum(axis=1)
    table = counts.copy().astype("string")
    for status in counts.columns:
        table[status] = [
            f"{count} ({100 * count / total:.1f}%)"
            for count, total in zip(counts[status], totals, strict=True)
        ]
    table.insert(0, "Exames", totals)
    display(table)

    percentages = counts.div(totals, axis=0).mul(100)
    ax = percentages.plot.barh(
        stacked=True,
        figsize=(10, 3.8),
        width=0.7,
        color=[colors[status] for status in counts.columns],
    )
    if show_counts:
        for status, bars in zip(counts.columns, ax.containers, strict=True):
            labels = [str(count) if count else "" for count in counts[status]]
            ax.bar_label(bars, labels=labels, label_type="center", fontsize=9)
    ax.set(xlabel="Exames (%)", ylabel="", title=title, xlim=(0, 100))
    ax.invert_yaxis()
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
    plt.show()
    return counts


def show_aorta_visual(resolution):
    if not VISUAL_AVAILABLE:
        print("Seção visual indisponível: confira a tabela de fontes.")
        return None
    summary = summarize_visual_status(
        assessments[resolution],
        "aorta_result",
        AORTA_ARTERY_STATUS_ORDER,
        DATASET_ORDER,
    )
    counts = summary.pivot(index="dataset", columns="status", values="count").reindex(
        index=DATASET_ORDER, columns=AORTA_ARTERY_STATUS_ORDER, fill_value=0
    )
    return show_aorta_distribution(
        counts,
        f"Avaliação visual da aorta — {resolution}",
        VISUAL_COLORS,
        show_counts=True,
    )

## 1. Avaliação visual da aorta

As classificações vêm das planilhas de inspeção visual. “Não avaliável” permanece separado de falha de segmentação. No OrCaScore high, TEV2P6 aparece na avaliação visual, embora não tenha resultado no CSV high.

Os números dentro das barras são quantidades de exames; o comprimento de cada segmento representa o percentual.

### Mid resolution

In [ ]:
aorta_mid = show_aorta_visual("mid")

### High resolution

In [ ]:
aorta_high = show_aorta_visual("high")

## 2. Dice da aorta no MM-WHS (treino)

O Dice de treino compara a predição com o rótulo 820 no volume processado. O Dice de teste usa uma grade de 1 mm e aparece na seção 3, com identificação do protocolo de avaliação.

In [ ]:
if MMWHS_NUMERIC_AVAILABLE:
    dice_by_case = pd.DataFrame(
        {
            resolution: load_mmwhs_train_aorta_dice(
                RUN_DIRS[resolution]["MM-WHS"] / "numeric/results_all.csv"
            )
            for resolution in RUN_DIRS
        }
    )
    if dice_by_case.isna().any().any():
        raise ValueError(
            "Os runs mid e high não possuem o mesmo conjunto de exames de treino"
        )
    dice_by_case.index.name = "Exame"
    print(f"Exames pareados com Dice: {len(dice_by_case)}")
else:
    print("Dice de treino indisponível: faltam os CSVs MM-WHS mid/high.")

### Resultado geral

In [ ]:
if MMWHS_NUMERIC_AVAILABLE:
    dice_overall = dice_by_case.agg(["count", "mean", "median", "std", "min", "max"]).T
    dice_overall.index.name = "Resolução"
    display(dice_overall.round(3))

    ax = dice_overall["mean"].plot.bar(
        color=[RESOLUTION_COLORS[resolution] for resolution in dice_overall.index],
        figsize=(6, 4),
        ylim=(0, 1),
        rot=0,
    )
    ax.set(
        xlabel="",
        ylabel="Dice médio da aorta",
        title="MM-WHS treino: Dice médio por resolução",
    )
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{bar.get_height():.3f}",
            ha="center",
        )
    plt.show()
else:
    print("Estatísticas de treino indisponíveis: faltam os CSVs MM-WHS.")

### Comparação por exame: Dice e classificações da aorta

O alerta automático e a avaliação visual aparecem lado a lado em cada resolução. Divergências mostram os limites da heurística; o Dice usa o ground truth e não é uma das entradas do alerta.

In [ ]:
if MMWHS_NUMERIC_AVAILABLE and VISUAL_AVAILABLE:
    visual_aorta = {}
    automatic_aorta = {}
    for resolution in ("mid", "high"):
        paired = (
            feedback_by_resolution[resolution]
            .loc[
                lambda frame: (
                    frame["dataset"].eq("MM-WHS") & frame["subset"].eq("train")
                )
            ]
            .set_index("image_id")
        )
        missing = dice_by_case.index.difference(paired.index)
        if len(missing):
            raise ValueError(
                f"{resolution}: {len(missing)} exames com Dice sem classificação da aorta: "
                f"{missing.tolist()}"
            )
        visual_aorta[resolution] = paired["aorta_result"].reindex(dice_by_case.index)
        automatic_aorta[resolution] = paired["feedback"].reindex(dice_by_case.index)

    dice_case_summary = pd.DataFrame(
        {
            "Dice mid": dice_by_case["mid"],
            "Aorta visual mid": visual_aorta["mid"],
            "Alerta automático mid": automatic_aorta["mid"],
            "Dice high": dice_by_case["high"],
            "Aorta visual high": visual_aorta["high"],
            "Alerta automático high": automatic_aorta["high"],
            "Delta Dice (high - mid)": dice_by_case["high"] - dice_by_case["mid"],
        }
    )
    display(dice_case_summary.round(4))
else:
    print("Comparação com avaliações visuais indisponível: confira as fontes.")

## 3. Dice da aorta no MM-WHS (teste)

O protocolo corrigido compara somente o label 820, com referência criptografada e reamostragem para 1 mm. As comparações exigem esse protocolo em mid e high. A primeira tabela apresenta o Dice geral (ausências contabilizadas como zero) e o Dice válido de cada resolução (sem avaliações ausentes). A segunda usa somente os mesmos exames válidos em ambas as resoluções: 34 nos runs selecionados. Dice zero efetivamente avaliado permanece em todas as análises; ausência não é confundida com esse resultado.

In [ ]:
if MMWHS_NUMERIC_AVAILABLE:
    test_dice_series = {
        resolution: load_mmwhs_test_aorta_dice(
            RUN_DIRS[resolution]["MM-WHS"] / "numeric/results_all.csv"
        )
        for resolution in ("mid", "high")
    }
    test_protocols = {
        series.attrs["aorta_evaluation_method"] for series in test_dice_series.values()
    }
    if test_protocols != {"mmwhs_aorta_label820_1mm_wine"}:
        raise ValueError(
            "Esperado protocolo de aorta isolada (label 820): não use resultados WHS legados."
        )
    print(
        "Protocolo do teste:", test_dice_series["mid"].attrs["aorta_evaluation_label"]
    )
    test_dice_by_case = pd.DataFrame(test_dice_series)
    if len(test_dice_by_case) != 40:
        raise ValueError("Esperados 40 exames CT test do MM-WHS")
    test_dice_by_case.index.name = "Exame"
    # Estatísticas válidas excluem ausências, mas preservam Dice zero avaliado.
    test_dice_overall = test_dice_by_case.agg(
        ["count", "mean", "median", "std", "min", "max"]
    ).T.rename(
        columns={
            "count": "Exames válidos",
            "mean": "Dice válido médio",
            "median": "Mediana válida",
            "std": "DP válido",
            "min": "Mínimo válido",
            "max": "Máximo válido",
        }
    )
    test_dice_overall.insert(
        0, "Dice geral (ausências = 0)", test_dice_by_case.fillna(0).mean()
    )
    test_dice_overall.insert(0, "total de exames", len(test_dice_by_case))
    test_dice_overall.index.name = "Resolução"
    display(test_dice_overall.round(4))
else:
    print("Dice de teste indisponível: faltam os CSVs MM-WHS mid/high.")

In [ ]:
# Compara somente os mesmos IDs com Dice disponível nas duas resoluções.
if MMWHS_NUMERIC_AVAILABLE:
    test_dice_common = test_dice_by_case.dropna(subset=["mid", "high"])
    test_dice_common_summary = test_dice_common.agg(
        ["count", "mean", "median", "std", "min", "max"]
    ).T.rename(
        columns={
            "count": "Exames comuns",
            "mean": "Dice comum médio",
            "median": "Mediana",
            "std": "DP",
            "min": "Mínimo",
            "max": "Máximo",
        }
    )
    test_dice_common_summary.index.name = "Resolução"
    print(
        f"Coorte pareada: {len(test_dice_common)} exames válidos em ambas as resoluções."
    )
    display(test_dice_common_summary.round(4))
else:
    print("Dice comum indisponível: faltam os CSVs MM-WHS mid/high.")

In [ ]:
if MMWHS_NUMERIC_AVAILABLE and VISUAL_AVAILABLE:
    test_visual_aorta = {}
    for resolution in ("mid", "high"):
        paired = (
            feedback_by_resolution[resolution]
            .loc[
                lambda frame: frame["dataset"].eq("MM-WHS") & frame["subset"].eq("test")
            ]
            .set_index("image_id")
        )
        if (
            paired.index.duplicated().any()
            or not test_dice_by_case.index.isin(paired.index).all()
        ):
            raise ValueError(
                f"{resolution}: avaliação visual de teste ausente ou duplicada"
            )
        test_visual_aorta[resolution] = paired["aorta_result"].reindex(
            test_dice_by_case.index
        )

    test_dice_case_summary = pd.DataFrame(
        {
            "Dice mid": test_dice_by_case["mid"],
            "Aorta visual mid": test_visual_aorta["mid"],
            "Dice high": test_dice_by_case["high"],
            "Aorta visual high": test_visual_aorta["high"],
            "Delta Dice (high - mid)": test_dice_by_case["high"]
            - test_dice_by_case["mid"],
        }
    )
    display(test_dice_case_summary.round(4))
else:
    print("Comparação visual de teste indisponível: confira as fontes.")

## 4. Visualização 3D de um exame representativo

Configure conjunto, resolução e ID no início. O padrão seleciona o treino `ct_train_1007` em Mid (Dice 0,9338), com máscara pública de referência. A predição persistida é carregada quando disponível, sem repetir o pipeline. Sem ID explícito, seleciona-se o caso próximo da mediana da coorte.

A referência de teste oficial é criptografada: não pode ser tratada como máscara anatômica. Sem uma referência legível, mostra-se somente a predição. Informe `QUALITATIVE_REFERENCE_PATH` com uma referência binária ou label 820 na mesma grade nativa da CT para obter referência, predição e sobreposição. No treino, o rótulo público é carregado automaticamente.

Uma única figura contém duas superfícies independentes: referência em verde (opacidade 0,35) e predição em vermelho (0,25). A visibilidade de cada superfície pode ser alternada no painel de objetos do K3D, preservando câmera, escala e enquadramento. A suavização leve atua apenas nos vértices da malha (duas iterações, intensidade 0,15), sem modificar máscaras ou Dice.

Nenhum resultado científico é regravado. O Dice salvo pode usar uma grade diferente da visualização nativa.


In [ ]:
if not SHOW_QUALITATIVE:
    print("3D desativado. Ative SHOW_QUALITATIVE na configuração.")
elif not MMWHS_NUMERIC_AVAILABLE:
    print("3D indisponível: confira as fontes MM-WHS.")
else:
    if QUALITATIVE_RESOLUTION not in ("mid", "high"):
        raise ValueError("QUALITATIVE_RESOLUTION deve ser mid ou high.")
    if MMWHS_DATA_ROOT is None:
        MMWHS_DATA_ROOT = resolve_existing_path(
            "MMWHS_BASE_PATH",
            [EXTERNAL_RESULTS_ROOT.parent / "DatasetsCCTA/MM-WHS-2017-Dataset"],
            "MM-WHS",
        )
    MMWHS_DATA_ROOT = Path(MMWHS_DATA_ROOT).expanduser()
    if not MMWHS_DATA_ROOT.is_dir():
        raise FileNotFoundError(
            "Informe MMWHS_DATA_ROOT ou MMWHS_BASE_PATH com o banco montado."
        )
    if QUALITATIVE_SUBSET not in ("train", "test"):
        raise ValueError("QUALITATIVE_SUBSET deve ser train ou test.")
    qual_table = test_dice_by_case if QUALITATIVE_SUBSET == "test" else dice_by_case
    qual_scores = qual_table[QUALITATIVE_RESOLUTION].dropna().sort_index()
    qual_exam_id = (
        QUALITATIVE_EXAM_ID or (qual_scores - qual_scores.median()).abs().idxmin()
    )
    if qual_exam_id not in qual_scores.index:
        raise ValueError(
            "Selecione um exame do conjunto escolhido com Dice registrado."
        )
    qual_inventory = discover_ccta_dataset("mmwhs", MMWHS_DATA_ROOT)
    qual_matches = qual_inventory.loc[
        qual_inventory["exam_id"].eq(qual_exam_id)
        & qual_inventory["subset"].eq(QUALITATIVE_SUBSET)
    ]
    if len(qual_matches) != 1:
        raise ValueError(f"Exame ausente ou duplicado: {qual_exam_id}")
    qual_record = qual_matches.iloc[0]
    qual_reference = load_ccta_aorta_ground_truth(qual_record)
    if QUALITATIVE_REFERENCE_PATH is not None:
        qual_ref_path = Path(QUALITATIVE_REFERENCE_PATH)
        if "encrypt" in qual_ref_path.name.lower():
            raise ValueError(
                "O rótulo criptografado não pode ser visualizado como referência."
            )
        qual_ref_nii = load_nifti(str(qual_ref_path))
        qual_image_nii = load_nifti(str(qual_record["path"]))
        if qual_ref_nii.shape != qual_image_nii.shape or not np.allclose(
            qual_ref_nii.affine, qual_image_nii.affine
        ):
            raise ValueError("A referência informada deve estar na grade nativa da CT.")
        qual_ref_values = np.asanyarray(qual_ref_nii.dataobj)
        if not np.isin(np.unique(qual_ref_values), [0, 1, 820]).all():
            # Referências multiclasse são aceitas somente com a classe 820.
            if not np.any(qual_ref_values == 820):
                raise ValueError("Referência sem label aórtico 820.")
        qual_reference = qual_ref_values == (
            820 if np.any(qual_ref_values == 820) else 1
        )
        if not np.any(qual_reference):
            raise ValueError("A referência informada está vazia.")
    if qual_reference is None:
        print(
            "Referência de teste criptografada: exibindo apenas a predição. "
            "Informe QUALITATIVE_REFERENCE_PATH para habilitar a sobreposição."
        )
    qual_run = RUN_DIRS[QUALITATIVE_RESOLUTION]["MM-WHS"]
    with (qual_run / "config/effective_pipeline_config.json").open() as handle:
        qual_config = json.load(handle)
    with (qual_run / "config/run_manifest.json").open() as handle:
        qual_manifest = json.load(handle)
    qual_saved_prediction = (
        qual_run
        / "evaluation/aorta"
        / QUALITATIVE_SUBSET
        / qual_exam_id
        / f"{qual_exam_id}_label.nii.gz"
    )
    print(
        f"{qual_exam_id} ({QUALITATIVE_RESOLUTION}); Dice salvo: {qual_scores.loc[qual_exam_id]:.4f}"
    )
    if qual_saved_prediction.is_file():
        validate_saved_prediction(Path(str(qual_record["path"])), qual_saved_prediction)
        qual_prediction = (
            np.asanyarray(load_nifti(str(qual_saved_prediction)).dataobj) == 820
        )
    else:
        print("Reconstruindo a máscara; não altera os resultados registrados.")
        qual_prediction = restore_native_mask(
            predict_aorta(
                qual_record,
                qual_config,
                align_volume=qual_manifest.get("orcascore_visual_alignment", True),
            )
        )
    if qual_reference is not None and qual_reference.shape != qual_prediction.shape:
        raise ValueError("Predição e referência possuem grades diferentes.")
    # Recorta a união das máscaras, sem excluir regiões de vazamento.
    qual_spacing = tuple(
        float(qual_record[c]) for c in ("spacing_x_mm", "spacing_y_mm", "spacing_z_mm")
    )
    qual_roi = mask_bounding_box_slices(
        (qual_prediction > 0)
        if qual_reference is None
        else qual_reference | (qual_prediction > 0),
        qual_spacing,
        3.0,
    )
    # As duas superfícies compartilham grade, câmera e enquadramento.
    qual_pred_view = np.transpose(qual_prediction[qual_roi], (1, 0, 2))
    qual_ref_view = (
        qual_pred_view
        if qual_reference is None
        else np.transpose(qual_reference[qual_roi], (1, 0, 2))
    )
    qual_view_spacing = (qual_spacing[1], qual_spacing[0], qual_spacing[2])
    qual_extent = (np.array(qual_pred_view.shape) - 1) * qual_view_spacing
    qual_center = qual_extent / 2
    qual_eye = qual_center + np.array([1.3, -1.6, 0.7]) * max(qual_extent.max(), 1.0)
    qual_camera = [*qual_eye, *qual_center, 0.0, 0.0, 1.0]
    qual_views = (
        {"overlay": (0.35, 0.25)}
        if qual_reference is not None
        else {"prediction": (0.0, 0.25)}
    )
    qual_plots = {}
    for qual_view, (ref_opacity, pred_opacity) in qual_views.items():
        qual_plots[qual_view] = visualize_binary_masks_comparison(
            qual_ref_view,
            qual_pred_view,
            spacing=qual_view_spacing,
            reference_name="Referência da aorta (820)"
            if qual_reference is not None
            else "Referência indisponível",
            predicted_name="Aorta segmentada",
            reference_color=0x22C55E,
            predicted_color=0xEF4444,
            reference_opacity=ref_opacity,
            predicted_opacity=pred_opacity,
            display_plot=False,
            mesh_smoothing_iterations=MESH_SMOOTHING_ITERATIONS,
            camera=qual_camera,
            plot_name=f"MM-WHS: {qual_exam_id} ({QUALITATIVE_RESOLUTION}) — {qual_view}",
        )
    qual_plot = next(iter(qual_plots.values()))
    print("Malhas prontas. Execute a próxima célula para exibir e exportar o 3D.")

### Exibir o exame e exportar HTML

Esta célula reutiliza `qual_plot`, sem repetir a segmentação. Se o widget K3D não aparecer no VS Code, abra no navegador o HTML independente indicado na saída. O arquivo é salvo na pasta `visual/` do run externo selecionado, não no repositório.

In [ ]:
if not SHOW_QUALITATIVE:
    print("3D desativado. Ative SHOW_QUALITATIVE na configuração.")
elif "qual_plot" not in globals():
    print("Execute primeiro a célula de reconstrução do exame.")
else:
    print(f"Exame: {qual_exam_id} | Resolução: {QUALITATIVE_RESOLUTION}")
    display(qual_plot)
    qual_html_path = qual_run / "visual" / f"{qual_exam_id}_aorta_overlay.html"
    save_k3d_plot_html(qual_plot, qual_html_path)
    print("HTML independente:", qual_html_path.resolve())